# NB_patient_360 — Gold Layer: Patient 360 View

Builds `caresyncdata.gold.patient_360` — one row per patient enriched with hospital name, insurance provider name, and aggregated lab-result stats.

**STTM:** Silver → Gold | CareSync Health Network 
**Load type:** Full rebuild — `CREATE OR REPLACE TABLE … AS SELECT` 
**Grain:** One row per patient 
**Driving table:** `caresyncdata.silver.patients` 
**Joins:** LEFT JOIN `hospitals`, `insurance_providers`, aggregated `lab_results`

> ⚠️ **Pre-requisite:** `caresyncdata.silver.patients` must exist before running this notebook. The `patients` table is currently available only in the bronze layer. Run the `landing_silver` notebook for `patients` first.

In [0]:
# ---------------------------------------------------------------------------
# Configuration — substituting the <catalog> placeholder from the STTM
# ---------------------------------------------------------------------------
catalog = "caresyncdata"
silver  = f"{catalog}.silver"
gold    = f"{catalog}.gold"
target_table = f"{gold}.patient_360"

print(f"Catalog  : {catalog}")
print(f"Source   : {silver}")
print(f"Target   : {target_table}")

In [0]:
# Ensure the gold schema exists before writing
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print(f"Schema '{gold}' is ready.")

In [0]:
# ---------------------------------------------------------------------------
# Full rebuild — CREATE OR REPLACE TABLE ... AS SELECT
# Grain: one row per patient
# Driving table: silver.patients
# Joins: LEFT JOIN hospitals, insurance_providers, aggregated lab_results
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH lab_agg AS (
    -- Aggregate lab results per patient
    SELECT
        patient_id,
        COUNT(*)                                                  AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)   AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)   AS critical_test_count,
        MAX(test_date)                                            AS last_test_date
    FROM {silver}.lab_results
    GROUP BY patient_id
)
SELECT
    -- Patient identity
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                                    AS date_of_birth,
    CAST(
        FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25)
    AS INT)                                                                           AS age,
    p.gender,
    p.city,
    p.state,

    -- Hospital (via LEFT JOIN)
    p.registered_hospital_id                                                         AS hospital_id,
    h.hospital_name,

    -- Insurance (via LEFT JOIN)
    p.insurance_provider_id,
    i.provider_name                                                                  AS insurance_provider_name,

    -- Lab summary (from CTE; COALESCE to 0 when no tests exist)
    COALESCE(la.total_tests,          0)                                             AS total_tests,
    COALESCE(la.abnormal_test_count,  0)                                             AS abnormal_test_count,
    COALESCE(la.critical_test_count,  0)                                             AS critical_test_count,
    la.last_test_date

FROM {silver}.patients                   AS p
LEFT JOIN {silver}.hospitals             AS h  ON p.registered_hospital_id   = h.hospital_id
LEFT JOIN {silver}.insurance_providers   AS i  ON p.insurance_provider_id    = i.insurance_provider_id
LEFT JOIN lab_agg                        AS la ON p.patient_id               = la.patient_id
""")
print(f"\u2705  {target_table} rebuilt successfully.")

In [0]:
# Quick row-count and sample check
row_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {target_table}").collect()[0]['cnt']
print(f"Total rows in {target_table}: {row_count}")

df = spark.table(target_table)
display(df.limit(10))

In [0]:
records_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {target_table}").collect()[0]['cnt']
dbutils.notebook.exit(str(records_count))